In [1]:
import numpy as np
import torch
import torchvision
from imgsticher import get_image
import time
import os
import cv2 as cv

In [2]:
data_dir = "photos/dataset_final"# data_dir = "drive/MyDrive/campycad/dataset_final"

class_names = [labels for labels in os.listdir(data_dir)]
lable = torch.arange(0, len(class_names))

lable_Dict = dict(zip(lable.tolist(),class_names))
Dict_lable = dict(zip(class_names, lable.tolist()))

In [3]:
num_img_in_training = 10
Tsum = 0
images=[]
img_data=[]
'''   [
      #img 1
      [{"boxes":[[x0,y0,x1,y1],[x0,y0,x1,y1]],
        "labels":[1,2]},
        
       {"boxes":[[x0,y0,x1,y1]],
        "labels":[3]],
      #img 2
      [{"boxes":[[x0,y0,x1,y1],[x0,y0,x1,y1]],
        "labels":[1,2]},
        
       {"boxes":[[x0,y0,x1,y1]],
        "labels":[3]],
      ]
'''
for i in range(1,num_img_in_training+1):
    print(f'{int(i*100/num_img_in_training)}%||{Tsum/i}',end='\r')
    s = time.time()
    x,y = get_image()# np.ndarray, [[[x0,y0,x1,y1],'ClassLabel'],[x0,y0,x1,y1],'ClassLabel']]
    e = time.time()
    Tsum += (e-s)
    images.append(x)
    this_img_data = {}# {'box':[[x0,y0,x1,y1],[x0,y0,x1,y1]],'label':[1,2] *25}
    l_boxes = []
    l_labels = []
    for j in range(len(y)):
        l_boxes.append(y[j][0])
        l_labels.append(Dict_lable[y[j][1]])
    this_img_data['boxes'] = torch.tensor(l_boxes,dtype=torch.float)
    this_img_data['labels'] = torch.tensor(l_labels,dtype=torch.int64)
    img_data.append(this_img_data)
#print(img_data)
#opencv display images created
print(f'{Tsum} secs for {num_img_in_training} image')
print(img_data[0]['labels'].dtype) #torch.float64

8.171980142593384 secs for 10 image
torch.int64


In [4]:
Images_final=[]
for i in range(len(images)):
    img_t = torch.from_numpy(images[i].transpose(2,0,1))
    Images_final.append(img_t.to(dtype=torch.float))
Images_final[0].dtype #torch.float64

torch.float32

In [5]:
del l_boxes,l_labels,Tsum,num_img_in_training,this_img_data,x,y,e,s,class_names,lable,i,j,Dict_lable,data_dir,img_t

In [6]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn()
output = model(Images_final,img_data)
model.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu

In [7]:
x,i = get_image()
y = x
# cv.imshow('ph',x)
print(i)
x = torch.from_numpy(x.transpose(2,0,1))
print(x.shape)
prediction = model([x.to(dtype=torch.float)])
print(prediction)
# cv.waitKey(0)

[[[11, 0, 131, 120], 'wire_diag_r0'], [[145, 0, 265, 120], 'battery_r2'], [[278, 0, 398, 120], 'dep_curr_src_r1'], [[409, 0, 529, 120], 'dep_curr_src_r0'], [[541, 0, 661, 120], 'cap_r0'], [[16, 718, 136, 838], 'dc_volt_src_2_r0'], [[150, 718, 270, 838], 'dep_curr_src_r2'], [[285, 718, 405, 838], 'battery_r0'], [[415, 718, 535, 838], 'dep_curr_src_r0'], [[551, 718, 671, 838], 'curr_src_r3'], [[16, 1430, 136, 1550], 'resistor_r1'], [[156, 1430, 276, 1550], 'curr_src_r1'], [[291, 1430, 411, 1550], 'gnd_1'], [[425, 1430, 545, 1550], 'ac_src_r1'], [[564, 1430, 684, 1550], 'curr_src_r2'], [[10, 2145, 130, 2265], 'wire_L_r0'], [[145, 2145, 265, 2265], 'wire_L_r2'], [[280, 2145, 400, 2265], 'battery_r0'], [[420, 2145, 540, 2265], 'diode_r3'], [[556, 2145, 676, 2265], 'diode_r1'], [[14, 2858, 134, 2978], 'battery_r0'], [[150, 2858, 270, 2978], 'dep_volt_r3'], [[280, 2858, 400, 2978], 'dep_curr_src_r0'], [[419, 2858, 539, 2978], 'wire_L_r2'], [[554, 2858, 674, 2978], 'dc_volt_src_2_r2']]
torch.S

In [24]:
for i in range(len(prediction[0]['boxes'])):
    num=int(prediction[0]['labels'][i])
    print(lable_Dict[num] if num<=44 else num)
    a = (round(prediction[0]['boxes'][i][0].item()),round(prediction[0]['boxes'][i][1].item()))
    b = (round(prediction[0]['boxes'][i][2].item()), round(prediction[0]['boxes'][i][3].item()))
    if prediction[0]['scores'][i]>=0.7:
        img = cv.rectangle(y,a,b,color=(255,0,0))
cv.imshow('uh',img)
cv.waitKey(0)
torch.save(model,'!mod.pth')

battery_r2
55
47
47
battery_r2
47
61
55
49
battery_r2
battery_r2
dc_volt_src_2_r2
47
dc_volt_src_2_r2
65
74
battery_r2
50
battery_r2
diode_r0
dc_volt_src_2_r2
wire_L_r3
70
diode_r1
diode_r0
dc_volt_src_2_r3
61
47
resistor_r0
65
60
47
65
wire_L_r3
47
diode_r0
diode_r0
60
85
resistor_r0
dep_volt_r2
74
61
diode_r0
inductor_r0
battery_r2
65
inductor_r0
battery_r2
85
47
inductor_r0
inductor_r0
diode_r1
battery_r2
battery_r2
47
diode_r0
56
battery_r2
47
wire_L_r3
74
diode_r3
dep_volt_r2
67
inductor_r0
60
56
wire_L_r1
inductor_r0
63
